In [2]:
# Jaxlib
import jax
from jax import lax
from jax import random as jrnd
from jax import numpy as jnp
jax.config.update('jax_enable_x64', True)

# Others
from matplotlib import pyplot as plt

# This
from numerics import *
#from seismic import *

In [ ]:
import jax
from jax import lax
from jax import numpy as jnp
from jax import random as jrnd
from jax.typing import ArrayLike

import polars as pl 
from importlib.resources import files

from seismic.gm_utils import *

gmc = pl.read_csv(files("seismic") / "CY14_coeffs.csv")
gmc[-2, 'T'] = -1.
gmc[-1, 'T'] = -2.
gmc = gmc.sort('T')
gmc_col = gmc.columns
gmc_df = gmc
print(gmc_df[7:12])
gmc = gmc.cast(pl.Float64).to_jax().T
T_CY = gmc[0]
empty = jnp.zeros_like(T_CY)

c_RB_all = gmc[4]
c_n_all, c_M_all, c_HM_all = gmc[jnp.array([12, 13, 16])]
# c1, c1a - d
c1_all, c1a_all, c1b_all, c1c_all, c1d_all = gmc[7:12]
# c2
c2_all, c3_all, c4_all, c4a_all = gmc[jnp.array([1, 14, 2, 3])]
c5_all, c6_all, c7_all, c7b_all = gmc[jnp.array([15, 17, 18, 19])]
c8_all, c8a_all, c8b_all = gmc[jnp.array([5, 6, 20])]
c9_all, c9a_all, c9b_all, c11_all, c11b_all = gmc[jnp.array([21, 22, 23, 24, 25])]
# c_gamma
c_gamma_all = jnp.insert(gmc[26:29], 0, empty, axis = 0)
c_phi_all = jnp.insert(gmc[29:35], 0, empty, axis = 0)
c_tau_all = jnp.insert(gmc[35:37], 0, empty, axis = 0)
c_sigma_all = jnp.insert(gmc[37:40], 0, empty, axis = 0)
c_sigma2_JP, c_gamma_JP_IT, c_gamma_WN = gmc[40:43]
c_phi1_JP, c_phi5_JP, c_phi6_JP = gmc[43:]

z_tor_const_RV = jnp.array([2.704, 1.226, 5.849])
z_tor_const_NM = jnp.array([2.673, 1.136, 4.97])

A = 571. ** 4
B = 1360. ** 4 + A

def slice_coeffs(T):
    T_idx = jnp.searchsorted(T_CY, T) - 1
    T_slice = lax.dynamic_slice_in_dim()
    return T_slice,

def f_SA_ref(Mw, dip, z_tor, SOF_flag,  R_jb, R_rup, R_x, ):
    dip_rad = jnp.deg2rad(dip)

    r1 = c[1, 0] + c[2, 0] * (Mw - 6.) + ((c[2, 0] - c[3, 0]) / c_n) * jnp.log(1 + jnp.exp(c_n * (c_M - Mw)))
    r2 = c[4, 0] * jnp.log(R_rup + c[5, 0] * jnp.cosh(c[6, 0] * jnp.maximum(Mw - c_HM, 0.)))
    gamma = c_gamma[1] + c_gamma[2] / jnp.cosh(jnp.maximum(Mw - c_gamma[3], 0.))
    r3 = (c[4, 1] - c[4, 0]) * jnp.log(jnp.sqrt(R_rup ** 2 + c_RB ** 2)) + R_rup * gamma

    cosh_Mw = jnp.cosh(2 * jnp.maximum(Mw - c_gamma[3], 0.))
    cos_dip = jnp.cos(dip_rad)

    # Calculate Mw_z_tor (separate fn in nshmp-haz)
    is_rev = SOF_flag == -1.
    Mw_z_tor_rev = lambda Mw: jnp.clip(2.704 - 1.226 *(Mw - 5.849) , min = 0., max = 2.704)
    Mw_z_tor_else = lambda Mw: jnp.clip(2.673 - 1.136 *(Mw - 4.970) , min = 0., max = 2.673)
    Mw_z_tor = lax.cond(is_rev, Mw_z_tor_rev, Mw_z_tor_else, Mw)

    dz_tor = z_tor - Mw_z_tor

    r4 = ((c[7, 0] + c[7, 2]) * dz_tor + (c[11, 0] + c[11, 2]) * cos_dip ** 2) / cosh_Mw
    r5_cond = R_x >= 0.
    r5 = c[9, 0] * jnp.cos(dip_rad) * \
        (c[9, 1] + (1 - c[9, 1]) * jnp.tanh(R_x / c[9, 2])) * \
        (1 - jnp.sqrt(R_jb ** 2 + z_tor ** 2) / (R_rup + 1.))
    r5 = jnp.where(r5_cond, r5, 0.)
    
    return jnp.exp(r1 + r2 + r3 + r4 + r5)

def f_lnSA(vs30, z1p0, SA_ref, soil_nonlin):
    soil_lin = c_phi[1] * jnp.minimum(jnp.log(vs30 / 1130.), 0.)
    soil_nonlin_mod = soil_nonlin * jnp.log((SA_ref + c_phi[4]) / c_phi[4])

    # Calculate delta z1p0 (separate fn in nshmp-haz)
    z1p0_ref = jnp.exp(-7.15 / 4 * jnp.log((vs30 ** 4 + A) / B))
    dz1p0 = (z1p0 * 1000) - z1p0_ref

    rk_depth = c_phi[5] * (1. - jnp.exp(-dz1p0 / c_phi[6]))

    return jnp.log(SA_ref) + soil_lin + soil_nonlin_mod + rk_depth

def f_std(Mw, vs30inf_flag, SA_ref, soil_nonlin):
    nonlin_0 = soil_nonlin * SA_ref / (SA_ref + c_phi[4])
    nonlin_0sq = (1 + nonlin_0) ** 2
    Mw_thresh = jnp.clip(Mw - 5., min = 0, max = 1.5)
    tau = c_tau[1] + (c_tau[2] - c_tau[1]) / 1.5 * Mw_thresh

    sig_nonlin_0 = c_sigma[1] + (c_sigma[2] - c_sigma[1]) / 1.5 * Mw_thresh
    vs_term = jnp.where(vs30inf_flag == 1., c_sigma[3], 0.7)
    sig_nonlin_0 = sig_nonlin_0 * jnp.sqrt(vs_term + nonlin_0sq)

    return (tau ** 2 * nonlin_0sq + sig_nonlin_0 ** 2)

def f_CY14(Mw:float, site:Site, fault:Fault, R:jax.Array):
    R_jb, R_rup, R_epi, R_hyp, R_x = R
    SA_ref = f_SA_ref(Mw, fault.dip, fault.z_tor, fault.calc_SOF_flag(), R_jb, R_rup, R_x)

    # Calculate nonlinear attenuation effect (separate fn in nshmp-haz)
    soil_nonlin1 = jnp.exp(c_phi[3] * (jnp.minimum(site.vs30, 1130.) - 360.))
    soil_nonlin2 = jnp.exp(c_phi[3] * (1130. - 360.))
    soil_nonlin = c_phi[2] * (soil_nonlin1 - soil_nonlin2)

    lnSA = f_lnSA(site.vs30, site.z1p0, SA_ref, soil_nonlin)
    std = f_std(Mw, site.vs30inf_flag, SA_ref, soil_nonlin)

    lnSA = jnp.interp(T_master, T[T_sort], lnSA[T_sort])
    std = jnp.interp(T_master, T[T_sort], std[T_sort])

    return lnSA, std


shape: (5, 46)
┌───────┬──────┬──────┬──────┬───┬──────────┬─────────┬─────────┬─────────┐
│ T     ┆ c2   ┆ c4   ┆ c4a  ┆ … ┆ gamma_WN ┆ phi1_JP ┆ phi5_JP ┆ phi6_JP │
│ ---   ┆ ---  ┆ ---  ┆ ---  ┆   ┆ ---      ┆ ---     ┆ ---     ┆ ---     │
│ str   ┆ f64  ┆ f64  ┆ f64  ┆   ┆ f64      ┆ f64     ┆ f64     ┆ i64     │
╞═══════╪══════╪══════╪══════╪═══╪══════════╪═════════╪═════════╪═════════╡
│ 0.075 ┆ 1.06 ┆ -2.1 ┆ -0.5 ┆ … ┆ 0.7956   ┆ -0.4685 ┆ 0.383   ┆ 800     │
│ 0.1   ┆ 1.06 ┆ -2.1 ┆ -0.5 ┆ … ┆ 0.7932   ┆ -0.4985 ┆ 0.375   ┆ 800     │
│ 0.15  ┆ 1.06 ┆ -2.1 ┆ -0.5 ┆ … ┆ 0.7437   ┆ -0.6451 ┆ 0.379   ┆ 800     │
│ 0.2   ┆ 1.06 ┆ -2.1 ┆ -0.5 ┆ … ┆ 0.6922   ┆ -0.7653 ┆ 0.384   ┆ 800     │
│ 0.25  ┆ 1.06 ┆ -2.1 ┆ -0.5 ┆ … ┆ 0.6579   ┆ -0.8469 ┆ 0.393   ┆ 800     │
└───────┴──────┴──────┴──────┴───┴──────────┴─────────┴─────────┴─────────┘
shape: (24, 1)
┌────────┐
│ c9     │
│ ---    │
│ f64    │
╞════════╡
│ 0.9228 │
│ 0.3079 │
│ 0.9228 │
│ 0.9296 │
│ 0.9396 │
│ …      │
│ 0.3917 │
│ 

In [3]:
Mw = 6.3
site = Site(0., 0., 760., 0.5, 2.9, 1.)
erf1 = ERF(9., 5., 5., 8.)
erf2 = ERF(9., 4., 4., 8.)
erf3 = ERF(10., 5., 4., 5.)
fault1 = Fault(0., 20., 3.0, 0.1, 40., 80., 90., 2.5, 0., erf1)
fault2 = Fault(0., 25., 0.5, 0.2, 80., 80., 30., 0.5, 1., erf2)
fault3 = Fault(22., 12.1, 3.9, 1.2, 130., 40., 22., 3., 1., erf3)
fault_tree = make_fault_tree(fault1, fault2, fault3)
scn = Scenario(site, fault_tree)

In [4]:
# xyz Distance from rupture surface
def calc_R_rup(site: Site, fault: Fault) -> float:
    """Calculate 3d distance between site and nearest part of rupture"""
    # Grab coordinates
    site_xyz = site.calc_xyz()
    fault_xyz_hyp = fault.calc_xyz_hyp()
    fault_dxyz_tor = fault.calc_dxyz_tor()
    # Overall distance between hypocenter and TOR
    fault_dr_tor = jnp.linalg.norm(fault_dxyz_tor, ord = 2)
    # Distance for down-dip edge (edge2)
    scaling = (fault.width - fault_dr_tor) / fault_dr_tor
    # Fault edges
    edge1_xyz = fault_xyz_hyp + fault_dxyz_tor
    edge2_xyz = fault_xyz_hyp - fault_dxyz_tor * scaling

    # Distance vectors
    edge1_dxyz = edge1_xyz - site_xyz
    edge2_dxyz = edge2_xyz - site_xyz
    span_xyz = edge2_xyz - edge1_xyz

    # Distances
    edge1_r = jnp.linalg.norm(edge1_dxyz, ord = 2)
    edge2_r = jnp.linalg.norm(edge2_dxyz, ord = 2)
    span_xyz_cross = jnp.linalg.cross(span_xyz, edge1_dxyz)
    span_r = jnp.linalg.norm(span_xyz_cross, ord = 2) / jnp.linalg.norm(span_xyz, ord = 2)
    proj_xyz = edge1_xyz + (edge1_dxyz @ span_xyz) / (span_xyz @ span_xyz) * span_xyz
    proj_z = proj_xyz[-1]

    # Condition for edge vs. other distance
    edge_r = jnp.minimum(edge1_r, edge2_r)
    edge_cond = jnp.logical_or(proj_z > edge2_xyz[-1], proj_z < fault.z_tor)
    return lax.select(edge_cond, edge_r, span_r)

# xy Distance from epicenter
def calc_R_epi(site: Site, fault: Fault) -> float:
    """Calculate 2d distance between site and hypocenter."""
    return jnp.linalg.norm(site.calc_xy() - fault.calc_xy_hyp(), ord=2)

# xyz Distance from hypocenter
def calc_R_hyp(site: Site, fault: Fault) -> float:
    """Calculate 3d distance between site and hypocenter."""
    return jnp.linalg.norm(site.calc_xyz() - fault.calc_xyz_hyp(), ord=2)

# xy Distance from top of rupture
def calc_R_x(site: Site, fault: Fault) -> float:
    """Calculate 2d distance between site and top of rupture."""
    fault_xyz_tor = fault.calc_xyz_hyp() + fault.calc_dxyz_tor()
    return jnp.linalg.norm(site.calc_xyz() - fault_xyz_tor, ord=2)

def calc_R(site:Site, fault:Fault):
    """Calculate distances of interest."""
    return [calc_R_jb(site, fault), calc_R_rup(site, fault), calc_R_epi(site, fault), calc_R_hyp(site, fault), calc_R_x(site, fault)]

In [5]:
from jax.scipy.stats.norm import cdf as gaussian_cdf
from time import time

@jtu.register_pytree_node_class
class GMMLT:
    def __init__(self, gmms:list, weights:jax.typing.ArrayLike):
        self.gmms = gmms
        self.weights = weights

    # Calculate for a single GMM. Takes R so we don't repeat the calculation every time.
    def calc_single(self, i:int, Mw:float, site:Site, fault:Fault, R:jax.Array):
        return lax.switch(i, self.gmms, Mw, site, fault, R)

    def tree_flatten(self):
        return (self.weights), self.gmms
    
    @classmethod
    def tree_unflatten(cls, aux, children):
        return cls(aux, *children)

def calc_haz(IM:float, M_min:float, gmms:GMMLT, scn:Scenario, dM:float = 0.1):
    t0 = time()
    fault_num = scn.fault_tree.x.shape[0]
    M_min, M_max = scn.fault_tree.erf.M_min, scn.fault_tree.erf.M_max

    # Midpoint quadrature over maximum range
    roots_M = jnp.arange(M_min.min() + dM / 2, M_max.max() + dM / 2, dM)
    # Array of ones/zeros for each fault signifying array inside/outside range (shape (roots_M.shape, fault_num))
    weights_mask = (roots_M[:, None] > M_min[None, :]) & (roots_M[:, None] < M_max[None, :])
    # Quadrature weights
    weights_M = ((M_max - M_min) / weights_mask.sum(axis = 0))
    weights_M = jnp.einsum('ij,j->ij', weights_mask, weights_M)

    # ERF incremental rates
    n_M = jax.vmap(scn.fault_tree.erf.calc_n)(roots_M)

    # Ground motion means + stds for each fault at Legendre roots
    # Calculate R so we don't need to do it every time
    R_tree = jax.vmap(calc_R, in_axes = (None, 0))(scn.site, scn.fault_tree)
    # Triple vmap. First, across faults (and corresponding distances)
    calc_faults = jax.vmap(gmms.calc_single, in_axes=(None, None, None, 0, 0))
    # Then across magnitudes
    calc_M = jax.vmap(calc_faults, in_axes=(None, 0, None, None, None))
    # Then across GMMs. This order minimizes recompilation
    calc_gmms = jax.vmap(calc_M, in_axes=(0, None, None, None, None))
    # Grab indices and vmap across
    gmm_idcs = jnp.arange(len(gmms.gmms))
    all_mu_lnSA, all_std_lnSA = calc_gmms(gmm_idcs, roots_M, scn.site, scn.fault_tree, R_tree)
    # And weight according to logic tree
    mu_lnSA = jnp.einsum('i,ijkl->jkl', gmms.weights, all_mu_lnSA)
    std_lnSA = jnp.einsum('i,ijkl->jkl', gmms.weights, all_std_lnSA)
    
    # Probabilities of exceedance 
    prob_IM = 1 - gaussian_cdf(jnp.log(IM), mu_lnSA, std_lnSA)
    # Hazard integrand (magnitude probabilities * exceedance probabilities)
    haz_intgrnd = jnp.einsum('ijk,ij->ijk', prob_IM, n_M)
    # Integrate using mag weights
    haz = jnp.einsum('ij,ijk->k', weights_M, haz_intgrnd)
    return haz

def calc_marginal_haz(IM:float, M_min:float, gmms:GMMLT, scn:Scenario, n:int = 30):
    pass 
    

    """
    Marginal hazard for a single scenario is:
    N_k * f_m(M) * f_r(R|M)*P(Y>IM|M,R)
    Where I'm assuming that f_r will be one for a point source...
    """

IM = 0.3
Mw_min = 4.
gmm_backbones = [f_ASK14, f_BSSA14, f_CB14, f_CY14, f_Idriss14]
gmm_epi_plus = []#[f_epistemic_AAY14(gmm, 3.18) for gmm in gmm_backbones]
gmm_epi_minus = []#[f_epistemic_AAY14(gmm, -3.18) for gmm in gmm_backbones]
gmms = gmm_backbones + gmm_epi_plus + gmm_epi_minus
weights = jnp.ones(len(gmms)) / len(gmms)
gmmlt = GMMLT(gmms, weights)
R = calc_R(site, fault1)
#print(gmmlt.calc_single(1, 5., site, fault1, R))
with jax.profiler.trace("/tmp/jax-trace", create_perfetto_trace=True):
    haz = calc_haz(IM, Mw_min, gmmlt, scn)
    haz.block_until_ready()

#plt.plot(T_master, haz)
#plt.xscale('log')

E1208 03:57:37.601718  145472 python_hooks.cc:416] Can't import tensorflow.python.profiler.trace
E1208 03:57:46.309141  145472 python_hooks.cc:416] Can't import tensorflow.python.profiler.trace


In [6]:
import os
os.listdir('/tmp/jax-trace/')

['.DS_Store', 'plugins']